## LangGraph Masterclass - Building Agentic AI Systems

## What You'll Learn

| # | Topic | Complexity |
|---|-------|------------|
| 1 | LangGraph Fundamentals — State, Nodes, Edges | ⭐ |
| 2 | Building Your First Graph | ⭐ |
| 3 | Conditional Routing & Branching | ⭐⭐ |
| 4 | Tool Integration with Google APIs | ⭐⭐ |
| 5 | Building a ReAct Agent | ⭐⭐⭐ |
| 6 | Memory & Checkpointing | ⭐⭐⭐ |
| 7 | Human-in-the-Loop Patterns | ⭐⭐⭐ |
| 8 | Subgraphs & Multi-Agent Orchestration | ⭐⭐⭐⭐ |
| 9 | Streaming & Real-time Output | ⭐⭐⭐ |
| 10 | Error Handling, Retries & Fallbacks | ⭐⭐⭐ |
| 11 | Full Production Agent — Putting It All Together | ⭐⭐⭐⭐⭐ |

## 0. Prerequisites & Installation

In [1]:
# Install required packages
!pip3 install -q langgraph langchain langchain-google-genai langchain-community google-api-python-client

In [2]:
import os
import getpass

# API Key Setup

# Option A: Set via environment variable (recommended for Cognizant environments)
#   export GOOGLE_API_KEY="your-gemini-api-key"
#
# Option B: Interactive prompt (for local dev / Colab)
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API Key: ")

# For Google Custom Search (Module 4 onwards)
# Get these from: https://programmablesearchengine.google.com/
if not os.environ.get("GOOGLE_CSE_ID"):
    os.environ["GOOGLE_CSE_ID"] = getpass.getpass("Enter your Google Custom Search Engine ID: ")

print("API keys configured.")

API keys configured.


In [3]:
# Verify Gemini Connectivity
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)
response = llm.invoke("Say 'LangGraph training is live!' in one line.")
print(response.content)
print("Gemini connection verified.")

[{'type': 'text', 'text': 'LangGraph training is live!', 'extras': {'signature': 'EjQKMgEMOdbHMN5zhWDEPGULDiOi3NHaWiSDqxKATvSjvyaG67DiDq6pTiGFLaUpeq3JVAnk'}}]
Gemini connection verified.


## 1. LangGraph Fundamentals

### What is LangGraph?

LangGraph is a framework built on top of LangChain for creating **stateful, multi-step, cyclical AI workflows** as directed graphs.

### Why LangGraph over plain LangChain?

| Feature | LangChain (LCEL) | LangGraph |
|---------|------------------|-----------|
| Execution Model | Linear chain / DAG | Cyclic graph (loops allowed) |
| State Management | Implicit via RunnablePassthrough | Explicit, typed state |
| Human-in-the-loop | Manual implementation | Built-in interrupt/resume |
| Memory/Checkpointing | Separate setup | Native checkpointing |
| Error Recovery | Try/catch | Retry from last checkpoint |
| Multi-agent | Complex to orchestrate | First-class subgraphs |

### Core Concepts

```
┌──────────────────────────────────────────────────────┐
│                    StateGraph                        │
│                                                      │
│   START ──► [Node A] ──► [Node B] ──► END           │
│                │              ▲                       │
│                └──────────────┘  (cycle / loop)      │
│                                                      │
│   State: { messages: [], context: "", ... }          │
└──────────────────────────────────────────────────────┘
```

- **State** — A TypedDict that flows through the graph. Every node reads and writes to it.
- **Node** — A Python function that takes state, does work, and returns state updates.
- **Edge** — Connects nodes. Can be unconditional or conditional.
- **START / END** — Special sentinel nodes marking entry and exit points.

## 2. Building your first Graph

In [13]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END

#Step 1. Define the State Schema
# This is the "data contract" for your graph.
# Every node receives this state and returns partial updates.

class SummaryState(TypedDict):
    topic: str #Input Topic
    summary: str #Generated Summary
    quality_score: int #Evaluation Score (1-10)
    feedback: str #Evaluation Feedback

In [ ]:
# ============================================================
# STEP 2: Define Node Functions
# ============================================================
# Each node is a regular Python function.
#   Input:  the full state dict
#   Output: a PARTIAL dict — only the keys you want to update.
#           LangGraph merges this dict into the existing state.

import json
import re


def parse_json_response(text: str) -> dict:
    """Extract a JSON object from an LLM response.

    LLMs often wrap JSON in markdown fences (```json ... ```) or add
    surrounding prose. This helper finds the first {...} block and parses it.
    Returns {} if no valid JSON is found.